In [1]:
# Test data repair scripts for NY

import sys
import os
from dotenv import load_dotenv, find_dotenv
import matplotlib.pyplot as plt
import time
import pandas as pd
import numpy as np
import json

load_dotenv(find_dotenv())

ROOT_PATH = os.getenv("ROOT_PATH")
MY_DATA_PATH = os.getenv("MY_DATA_PATH")
RAW_DATA_PATH = os.getenv("RAW_DATA_PATH")
DEWEY_PATH = os.path.join(RAW_DATA_PATH, "dewey-downloads", "building-permits-united-states")

sys.path.append(os.path.join(ROOT_PATH, "scripts"))
import data_utils as du

sys.path.append(os.path.join(ROOT_PATH, "agent/scripts"))

from burbank_data_repair import data_repair
MY_JURISDICTION = "Burbank"

INPUT_FILEPATH = os.path.join(MY_DATA_PATH, "processed_data", "permits_la_sample.parquet")


In [2]:
df = pd.read_parquet(INPUT_FILEPATH)
sub_df = df[df["JURISDICTION"] == MY_JURISDICTION]

for col in ['FILE_DATE', 'PERMIT_DATE', 'FINAL_DATE']:
    sub_df[f'{col}_FLAG'] = ""

#sub_df_filled = sub_df.copy()
sub_df_filled = data_repair(sub_df)

assert(len(sub_df) == len(sub_df_filled))


In [3]:
print(f"FILE_DATE available (all): {sub_df['FILE_DATE'].notna().mean():.1%} -> {sub_df_filled['FILE_DATE'].notna().mean():.1%}")

print(f"PERMIT_DATE available (all): {sub_df['PERMIT_DATE'].notna().mean():.1%} -> {sub_df_filled['PERMIT_DATE'].notna().mean():.1%}")

print(f"FINAL_DATE available (all): {sub_df['FINAL_DATE'].notna().mean():.1%} -> {sub_df_filled['FINAL_DATE'].notna().mean():.1%}")

mask1 = sub_df['STATUS_NORMALIZED'].isin(['Active', 'Final'])
mask2 = sub_df_filled['STATUS_NORMALIZED'].isin(['Active', 'Final'])
print(f"PERMIT_DATE available (active/final): {sub_df.loc[mask1]['PERMIT_DATE'].notna().mean():.1%} -> {sub_df_filled.loc[mask2]['PERMIT_DATE'].notna().mean():.1%}")

mask1 = sub_df['STATUS_NORMALIZED'].isin(['Final'])
mask2 = sub_df_filled['STATUS_NORMALIZED'].isin(['Final'])
print(f"FINAL_DATE available (final): {sub_df.loc[mask1]['FINAL_DATE'].notna().mean():.1%} -> {sub_df_filled.loc[mask2]['FINAL_DATE'].notna().mean():.1%}")


FILE_DATE available (all): 99.5% -> 99.5%
PERMIT_DATE available (all): 55.6% -> 61.3%
FINAL_DATE available (all): 26.1% -> 26.1%
PERMIT_DATE available (active/final): 89.4% -> 91.4%
FINAL_DATE available (final): 99.7% -> 99.8%


In [4]:
print(sub_df['STATUS_NORMALIZED'].value_counts())
print(sub_df_filled['STATUS_NORMALIZED'].value_counts())

STATUS_NORMALIZED
Inactive     1148
Final         336
In Review     285
Active        230
Name: count, dtype: int64
STATUS_NORMALIZED
Inactive     1169
Final         460
Active        228
In Review     144
Name: count, dtype: int64


In [12]:
mask = sub_df_filled["PERMIT_DATE"].isna() & (sub_df_filled["STATUS_NORMALIZED"] == "Active")
#mask = sub_df_filled["JURISDICTION"].notna()
sample = sub_df_filled.loc[mask].sample(1).iloc[0]
DATA = sample["DATA"]
DATES_DATA = du.extract_date_fields(DATA) 

print(f"STATUS_NORMALIZED: {sample['STATUS_NORMALIZED']}    *Filled: {sample['STATUS_NORMALIZED_FLAG']}*")
print(f"RECORD_TYPE_ORIGINAL: {sample['RECORD_TYPE_ORIGINAL']}")
print(f"FILE_DATE: {sample['FILE_DATE']}       *Filled: {sample['FILE_DATE_FLAG']}*")
print(f"PERMIT_DATE: {sample['PERMIT_DATE']}   *Filled: {sample['PERMIT_DATE_FLAG']}*")
print(f"FINAL_DATE: {sample['FINAL_DATE']}     *Filled: {sample['FINAL_DATE_FLAG']}*")

print("DATES_DATA: ")
print(json.dumps(DATES_DATA, indent=2))



STATUS_NORMALIZED: Active    *Filled: nan*
RECORD_TYPE_ORIGINAL: Commercial Rental
FILE_DATE: 1985-06-25       *Filled: nan*
PERMIT_DATE: None   *Filled: nan*
FINAL_DATE: None     *Filled: nan*
DATES_DATA: 
{
  "Completed": "",
  "Issue Date": "",
  "Applied Date": "06/25/1985",
  "Permit Status": "Paid / Current"
}


In [6]:
print("DATA:")
print(json.dumps(json.loads(DATA), indent=2))



DATA:
{
  "Address": "502 , N BEL AIRE DR",
  "Permit #": "BS1002347",
  "Sub Type": "Supplemental Permit",
  "Completed": "01/11/2011",
  "Inspection": [
    {
      "Inspection Date": "09/09/2010",
      "Inspection Action": "Other Building",
      "Inspection Status": "Approved"
    },
    {
      "Inspection Date": "10/15/2010",
      "Inspection Action": "Other Building",
      "Inspection Status": "Approved"
    },
    {
      "Inspection Date": "01/11/2011",
      "Inspection Action": "Final Building",
      "Inspection Status": "Finaled"
    }
  ],
  "Issue Date": "04/13/2010",
  "Description": "Chimney repair above roof line. Subject to field inspection",
  "Permit Type": "Building Combo",
  "Applied Date": "04/13/2010",
  "Permit Status": "Permit Final",
  "People Information": []
}
